In [ ]:
import geopandas as gpd

In [ ]:
g  = gpd.read_file("../Data/Shapefiles/APES_metric.gpkg", layer="nuts3")
g = g.loc[(~g.geometry.is_empty) & (g["RecordType"] != "ZIKV")].copy()







y_vec =fin_m["incidence"].to_numpy().reshape(-1, 1)

for thr in thresholds:
    # 1) Build W at this threshold (row-standardize so rows sum to 1)
    W = DistanceBand(coords, threshold=thr, binary=True, silence_warnings=True)
    W.transform = "R"  # row-standardize

    # 2) Spatial lag of y as neighbor AVERAGE (thanks to transform='R')
    y_lag = lag_spatial(W, y_vec)
    #y_lag = W.sparse.dot(y).ravel()

    # 3) z-score the spatial lag so the intercept ~ 0 and effects are SD(Y) per SD(X)
    y_mean = y_lag.mean()
    y_std  = y_lag.std(ddof=0)
    if y_std == 0:
        print(f"[WARN] {thr/1000:.0f} km → zero variance in Wy; skipping.")
        continue
    y_lag_z = (y_lag - y_mean) / y_std

    # 4) Create flat DataFrame for statsmodels (same as your OLS)
    tmp = fin_m.copy()
    tmp["Y_weighted"] = y_lag_z
    Xy_thr = pd.DataFrame(tmp.drop(columns=[c for c in ["geometry", "cent"] if c in tmp.columns]))

    # 5) Fit OLS with your same formula
    lm_fit_thr = smf.ols(formula, data=Xy_thr).fit()

    # 6) Coefficient table (same formatting)
    coefs = lm_fit_thr.summary2().tables[1].reset_index().rename(columns={"index": "term"})
    coefs["estimate"]   = coefs["Coef."].round(2)
    coefs["std.error"]  = coefs["Std.Err."].round(2)
    coefs["p-value"]    = coefs["P>|t|"].apply(fmt_p)

    lm_table = coefs.loc[:, ["term", "estimate", "std.error", "p-value"]]
    lm_table = lm_table.rename(columns={"term": "Covariate", "estimate": "Estimate", "std.error": "SE"})

    # 7) Print nicely
    print(f"\n=== Distance threshold = {thr/1000:.0f} km (row-std W; z-scored Wy) ===")
    print(lm_table.to_string(index=False))
